# Pew silicon-sampling: breadth-vs-data fine-tuning on 2 x T4
Settings: Accelerator **GPU T4 x2**, Internet **On**, Add Input -> the private dataset containing `kaggle_pkg2/` (scripts, `adapter/`, `repo/`).
Run cells 1-4 interactively first (smoke test). Do NOT press "Save & Run All" until the smoke test passed.

In [ ]:
# 1. install + GPU check
!pip install -q -U bitsandbytes peft accelerate "transformers>=4.45"
import torch, bitsandbytes, peft, transformers
print("torch", torch.__version__, "| bnb", bitsandbytes.__version__, "| peft", peft.__version__, "| transformers", transformers.__version__)
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1

In [ ]:
# 2. locate package, build repo, resolve paths
import os, glob, shutil, subprocess, sys, time, json
os.environ["HF_HOME"] = "/tmp/hf"; os.makedirs("/tmp/hf", exist_ok=True)
def find(name, kind="file"):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"{name} not found under /kaggle/input. Attach the dataset (Add Input). Seen: {os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else None}"
    return hits[0]
PKG   = os.path.dirname(find("kaggle_pew_train.py"))
INFER, TRAIN, P0, EVAL = (f"{PKG}/{n}" for n in ("kaggle_pew_infer.py", "kaggle_pew_train.py", "kaggle_p0.py", "kaggle_eval.py"))
LABAD = os.path.dirname(find("adapter_config.json"))
src = os.path.dirname(os.path.dirname(find("repo/src/config.py")))
REPO = "/tmp/repo"
if not os.path.exists(f"{REPO}/src/config.py"):
    shutil.copytree(src, REPO)
for f in ("data/processed/pew_india_2021.parquet", "data/processed/pew_folds.json", "data/processed/pew_selected_items.json", "data/reference/pew_codebook.json"):
    assert os.path.exists(f"{REPO}/{f}"), f"missing {f}"
print("PKG", PKG, "\nREPO", REPO, "\nLAB adapter", LABAD)

In [ ]:
# 3. download base model (~15 GB, to /tmp/hf)
from huggingface_hub import snapshot_download
print(snapshot_download("Qwen/Qwen2.5-7B-Instruct", allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json"]))

In [ ]:
# 4. SMOKE TEST: dry-run + 6 real optimizer steps on one GPU. Read s_per_step: full run = 966 steps x s_per_step.
r = subprocess.run([sys.executable, TRAIN, "--repo", REPO, "--out-dir", "/kaggle/working/smoke", "--item-range", "0:24",
                    "--n-resp", "200", "--max-steps", "6", "--save-every", "1000"],
                   env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"}, capture_output=True, text=True)
print(r.stdout[-1800:]); print(r.stderr[-800:])
assert r.returncode == 0, "smoke training failed"
import glob as _g
rec = [json.loads(l) for l in open("/kaggle/working/smoke/train_log.jsonl")]
print(rec)

In [ ]:
# 5. RUN (call run_seed(0); later run_seed(1) for a second seed if quota remains)
def launch(cmd, gpu, log):
    return subprocess.Popen([sys.executable] + cmd, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)},
                            stdout=open(log, "a"), stderr=subprocess.STDOUT)

def wait(procs, logs, every=300):
    while any(p.poll() is None for p in procs):
        time.sleep(every)
        for lg in logs:
            print(os.path.basename(lg), "|", open(lg).read().splitlines()[-1][:200] if os.path.getsize(lg) else "", flush=True)
    print("exit codes:", [p.returncode for p in procs])
    assert all(p.returncode == 0 for p in procs), "a job failed; see logs"

def run_seed(seed, n_resp=2000, max_hours=5.0):
    W = f"/kaggle/working/seed{seed}"; os.makedirs(W, exist_ok=True)
    # --- phase 1: train B (8 items, GPU0) and A (24 items, GPU1) on matched example budgets
    common = ["--repo", REPO, "--n-resp", str(n_resp), "--seed", str(seed), "--max-hours", str(max_hours)]
    pB = launch([TRAIN, *common, "--out-dir", f"{W}/train_B", "--item-range", "0:8"], 0, f"{W}/log_train_B.txt"); time.sleep(120)
    pA = launch([TRAIN, *common, "--out-dir", f"{W}/train_A", "--item-range", "0:24"], 1, f"{W}/log_train_A.txt")
    wait([pB, pA], [f"{W}/log_train_B.txt", f"{W}/log_train_A.txt"])
    adB, adA = f"{W}/train_B/adapter_final", f"{W}/train_A/adapter_final"
    assert os.path.exists(adB) and os.path.exists(adA), "training did not finish (time budget?); re-run run_seed to resume"
    # --- phase 2: held-out evaluation, items 0:40 on 500 fold-0 test respondents
    ev = ["--repo", REPO, "--item-range", "0:40", "--n-resp", "500", "--nshards", "1", "--shard", "0", "--batch", "16", "--max-hours", "8"]
    e1 = launch([INFER, *ev, "--adapter", adB, "--out-dir", f"{W}/eval_B", "--conditions", "adapter_raw"], 0, f"{W}/log_eval_B.txt"); time.sleep(120)
    e2 = launch([INFER, *ev, "--adapter", adA, "--out-dir", f"{W}/eval_A", "--conditions", "adapter_raw"], 1, f"{W}/log_eval_A.txt")
    wait([e1, e2], [f"{W}/log_eval_B.txt", f"{W}/log_eval_A.txt"])
    if seed == 0:   # zero-shot base on the same rows, split over both GPUs (base is adapter-independent)
        ev2 = [a for a in ev if a not in ("--nshards", "1", "--shard", "0")]
        b = [launch([INFER, *ev2, "--nshards", "2", "--shard", str(s), "--adapter", adA, "--out-dir", f"{W}/eval_base",
                     "--conditions", "base_raw"], s, f"{W}/log_eval_base{s}.txt") for s in (0, 1)]
        wait(b, [f"{W}/log_eval_base0.txt", f"{W}/log_eval_base1.txt"])
    # --- phase 3: no-profile (P0) control for all adapters and base, 40 prompts only
    p = launch([P0, "--repo", REPO, "--adapters", f"lab={LABAD},A={adA},B={adB}", "--out", f"{W}/p0.json", "--item-range", "0:40"], 0, f"{W}/log_p0.txt")
    wait([p], [f"{W}/log_p0.txt"], every=30)
    shutil.make_archive(f"/kaggle/working/seed{seed}_results", "zip", W, ".")
    print("zip:", round(os.path.getsize(f"/kaggle/working/seed{seed}_results.zip") / 1e6, 1), "MB")

In [ ]:
# 6. go. (If the session dies, re-run cells 1-3 and this cell: training resumes from the last checkpoint.)
run_seed(0)

In [ ]:
# 7. optional second seed (different respondent pool and sample)
# run_seed(1)